# 6 — Figuras para o Artigo

**Pré-requisito:** `data/processed/results.pkl` gerado pelo notebook 4.

**Saídas:** `template-latex/figuras/pipeline_metodologico.png`, `template-latex/figuras/tipos_grafos.png`, tabela LaTeX impressa no stdout.

In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

warnings.filterwarnings('ignore')

ROOT_DIR      = Path('.')
RESULTS_PATH  = ROOT_DIR / 'data' / 'processed' / 'results.pkl'
LATEX_FIG_DIR = ROOT_DIR / 'template-latex' / 'figuras'
LATEX_FIG_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_PATH, 'rb') as f:
    saved = pickle.load(f)
results = saved['results']

print('Setup concluído.')

## Pipeline Metodológico

In [ ]:
def draw_pipeline_figure(save_path):
    fig, ax = plt.subplots(figsize=(16, 3.5))
    ax.set_xlim(-0.5, 15.5)
    ax.set_ylim(0, 3.5)
    ax.axis('off')

    steps = [
        ('Grafos\n(DIMACS +\nsintéticos)',          1.0,  '#AED6F1'),
        ('Rotulação\n(backtracking\nexato)',         3.1,  '#A9DFBF'),
        ('Extração de\nCaracterísticas',              5.2,  '#A9DFBF'),
        ('Normalização\n(StandardScaler\npor fold)', 7.3,  '#FAD7A0'),
        ('SMOTE\n(por fold)',                         9.2,  '#FAD7A0'),
        ('Grid Search\n(3-Fold CV\ninterno)',        11.1, '#F9E79F'),
        ('Avaliação\n(5-Fold CV\nexterno)',          13.0, '#D7BDE2'),
        ('Métricas\n+ Análise',                       14.9, '#D5D8DC'),
    ]

    BOX_W, BOX_H, CY = 1.7, 1.8, 1.8

    for i, (label, cx, color) in enumerate(steps):
        rect = plt.Rectangle((cx - BOX_W/2, CY - BOX_H/2), BOX_W, BOX_H,
                               linewidth=1.2, edgecolor='#444', facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(cx, CY, label, ha='center', va='center',
                fontsize=7.5, fontweight='bold', zorder=3, linespacing=1.4)
        if i < len(steps) - 1:
            next_cx = steps[i + 1][1]
            ax.annotate('',
                        xy=(next_cx - BOX_W/2 - 0.05, CY),
                        xytext=(cx + BOX_W/2 + 0.05, CY),
                        arrowprops=dict(arrowstyle='->', color='#333', lw=1.5),
                        zorder=4)

    ax.annotate('',
                xy=(steps[3][1] - BOX_W/2, CY - BOX_H/2 - 0.2),
                xytext=(steps[6][1] + BOX_W/2, CY - BOX_H/2 - 0.2),
                arrowprops=dict(arrowstyle='<->', color='#888', lw=1.2, linestyle='dashed'))
    ax.text((steps[3][1] + steps[6][1]) / 2, CY - BOX_H/2 - 0.42,
            'repetido em cada fold externo (5x)',
            ha='center', fontsize=7, color='#666')

    ax.set_title('Pipeline Metodológico', fontsize=12, fontweight='bold', pad=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Figura salva: {save_path}')


draw_pipeline_figure(LATEX_FIG_DIR / 'pipeline_metodologico.png')

## Tipos de Grafos

In [ ]:
def draw_graph_types_figure(save_path):
    examples = [
        ('(a) Aleatório',           nx.erdos_renyi_graph(10, 0.4, seed=42)),
        ('(b) Regular',             nx.random_regular_graph(3, 10, seed=42)),
        ('(c) Malha Planar',        nx.convert_node_labels_to_integers(nx.grid_2d_graph(3, 3))),
        ('(d) Clique $K_5$',        nx.complete_graph(5)),
        ('(e) Bipartido $K_{4,4}$', nx.complete_bipartite_graph(4, 4)),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
    for ax, (title, G) in zip(axes, examples):
        if 'Bipartido' in title:
            pos = nx.bipartite_layout(G, nodes=list(range(4)))
        else:
            pos = nx.spring_layout(G, seed=42)
        nx.draw_networkx(G, pos=pos, ax=ax, with_labels=False,
                         node_color='#2196F3', node_size=120,
                         edge_color='#555', width=1.2)
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Figura salva: {save_path}')


draw_graph_types_figure(LATEX_FIG_DIR / 'tipos_grafos.png')

## Tabela de Resultados em LaTeX

In [ ]:
rows = []
for name, m in results.items():
    def fmt(key):
        return f'{np.mean(m[key]):.3f} \u00b1 {np.std(m[key]):.3f}'
    rows.append({
        'Modelo'   : name,
        'Acurácia' : fmt('accuracy'),
        'Precisão' : fmt('precision'),
        'Revocação': fmt('recall'),
        'F1'       : fmt('f1'),
    })

results_df = pd.DataFrame(rows).set_index('Modelo')

latex_table = results_df.to_latex(
    caption='Desempenho médio (\u00b1 desvio padrão) — 5-Fold CV estratificado',
    label='tab:resultados',
    escape=False,
)
print(latex_table)